# Anti-Spoiler RAG Eval
**Position-bounded retrieval vs. unbounded baseline**

This notebook tests whether restricting an LLM reading assistant to only text the reader has already encountered prevents spoilers,
using *Pride and Prejudice* by Jane Austen as a resource.

## 0. Dependencies

In [7]:
import subprocess, sys
pkgs = ["sentence-transformers", "faiss-cpu", "anthropic", "openai", "pandas", "numpy"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)
print("Dependencies OK")


Dependencies OK


## 1. Configuration
Defining base variables + setting up LLM use. (I used a Claude API key with haiku as a base model for less tokens.)

We do embeddings using a model from HuggingFace. Reader is set to be in chapter 15.


In [8]:
from google.colab import userdata

# ── LLM BACKEND ────────────────────────────────────────────────────────────
BACKEND         = "anthropic"            # "anthropic" | "openai" | "ollama"
API_KEY         = userdata.get('API_KEY')
ANTHROPIC_MODEL = "claude-haiku-4-5"
OPENAI_MODEL    = "gpt-4o"
OLLAMA_MODEL    = "llama3.2"             # as shown by `ollama list`
OLLAMA_BASE_URL = "http://localhost:11434"

HF_TOKEN = userdata.get('HF_TOKEN')

import os
if API_KEY == "sk-...":
    API_KEY = None   # falls back to ANTHROPIC_API_KEY / OPENAI_API_KEY env vars

# ── BOOK ───────────────────────────────────────────────────────────────────
BOOK_URL    = "https://raw.githubusercontent.com/GITenberg/Pride-and-Prejudice_1342/master/1342.txt"
BOOK_TITLE  = "Pride and Prejudice"
BOOK_AUTHOR = "Jane Austen"

# Simulated reader position: chapters 1–N are "already read"
# Pride and Prejudice has 61 chapters; 15 = ~25% through the book
READER_POSITION = 15

# ── EVAL ───────────────────────────────────────────────────────────────────
N_QUESTIONS_PER_TIER = 4   # × 3 tiers = 12 questions total

# ── RETRIEVAL ──────────────────────────────────────────────────────────────
EMBED_MODEL = "all-MiniLM-L6-v2"   # fast 384-dim model; no API key required
TOP_K       = 10                     # retrieved chunks per query

# ───────────────────────────────────────────────────────────────────────────
print(f"Config  backend={BACKEND}  book='{BOOK_TITLE}'  reader_pos=chapter {READER_POSITION}")


Config  backend=anthropic  book='Pride and Prejudice'  reader_pos=chapter 15


## 2. Model-agnostic LLM client
A thin wrapper so the rest of the notebook never imports a vendor SDK directly. We can swap backends in Section 1 without touching anything below.


In [9]:
import os

class LLMClient:
    """
    Thin abstraction over Anthropic / OpenAI / Ollama.
    Usage: client.complete(system_prompt, user_prompt) -> str
    """
    def __init__(self, backend: str, api_key: str | None = None):
        self.backend = backend.lower()
        if self.backend == "anthropic":
            import anthropic
            key = api_key or os.environ.get("ANTHROPIC_API_KEY")
            self._client = anthropic.Anthropic(api_key=key)
            self._model  = ANTHROPIC_MODEL
        elif self.backend == "openai":
            import openai
            key = api_key or os.environ.get("OPENAI_API_KEY")
            self._client = openai.OpenAI(api_key=key)
            self._model  = OPENAI_MODEL
        elif self.backend == "ollama":
            import openai  # Ollama exposes an OpenAI-compatible /v1 endpoint
            self._client = openai.OpenAI(api_key="ollama", base_url=f"{OLLAMA_BASE_URL}/v1")
            self._model  = OLLAMA_MODEL
        else:
            raise ValueError(f"Unknown backend {backend!r}. Choose anthropic | openai | ollama")

    def complete(self, system: str, user: str, max_tokens: int = 1024) -> str:
        if self.backend == "anthropic":
            msg = self._client.messages.create(
                model=self._model, max_tokens=max_tokens,
                system=system, messages=[{"role": "user", "content": user}]
            )
            return msg.content[0].text.strip()
        else:  # openai-compatible (openai + ollama)
            resp = self._client.chat.completions.create(
                model=self._model, max_tokens=max_tokens,
                messages=[{"role": "system", "content": system},
                          {"role": "user",   "content": user}]
            )
            return resp.choices[0].message.content.strip()


llm = LLMClient(backend=BACKEND, api_key=API_KEY)
print(f"LLMClient ready  ({BACKEND} / {llm._model})")


LLMClient ready  (anthropic / claude-haiku-4-5)


## 3. Fetch the book and chunk by chapter
Each chunk carries:
- `chapter_index` - monotonic 1-based position (the anti-spoiler axis)
- `chapter_label` - human-readable label ("Chapter 5")
- `paragraph_index` - the index of the pagraph inside the chapter
- `text` - the paragraph body

Note that if a paragraph is too small, we merge it with the next paragraph until a minimum length of 400 is reached, in order not to avoid having tiny chunks.

In [46]:
import re
import urllib.request
from dataclasses import dataclass

@dataclass
class Chunk:
    chunk_id: str           # e.g. "ch15_p07"
    chapter_index: int      # 1-based, the position axis
    chapter_label: str      # "Chapter XV"
    paragraph_index: int    # 1-based within chapter
    text: str

MIN_CHARS = 400   # below this, merge with next
MAX_CHARS = 2000  # soft ceiling; don't split paragraphs to hit it

def _split_chapter_body(body: str, chapter_index: int, chapter_label: str) -> list[Chunk]:
    paragraphs = [p.strip() for p in re.split(r"\r?\n\s*\r?\n", body) if p.strip()]

    # Greedy merge of short paragraphs
    merged: list[str] = []
    buf = ""
    for p in paragraphs:
        if not buf:
            buf = p
        elif len(buf) < MIN_CHARS:
            buf = buf + "\n\n" + p
        else:
            merged.append(buf)
            buf = p
    if buf:
        merged.append(buf)

    return [
        Chunk(
            chunk_id=f"ch{chapter_index:02d}_p{i:02d}",
            chapter_index=chapter_index,
            chapter_label=chapter_label,
            paragraph_index=i,
            text=text,
        )
        for i, text in enumerate(merged, start=1)
    ]

def fetch_and_chunk(url: str) -> list[Chunk]:
    req = urllib.request.Request(url, headers={"User-Agent": "research-notebook/1.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        raw = r.read().decode("utf-8")

    # Split on "Chapter N" / "Chapter I" markers
    pattern = r"\r?\n(Chapter [\w]+\.?)\r?\n"
    parts   = re.split(pattern, raw)

    chunks, idx = [], 0
    i = 1
    while i < len(parts) - 1:
        label = parts[i].strip()
        body  = parts[i + 1].strip()
        if body:
            idx += 1
            chunks.extend(_split_chapter_body(body, idx, label))
        i += 2
    return chunks


chunks = fetch_and_chunk(BOOK_URL)
print(f"Loaded  '{BOOK_TITLE}' by {BOOK_AUTHOR}")
print(f"  Chunks  : {len(chunks)}")
chunks_chapter_1 = [c for c in chunks if c.chapter_index == 1]
print(f"Chunks in chapter 1: {len(chunks_chapter_1)}")

from statistics import mean, median
lengths = [len(c.text) for c in chunks]
print(f"  Chunk chars — min: {min(lengths)}, median: {median(lengths):.0f}, mean: {mean(lengths):.0f}, max: {max(lengths)}")

from collections import Counter
per_chapter = Counter(c.chapter_index for c in chunks)
print(f"  Chunks/chapter — min: {min(per_chapter.values())}, median: {median(per_chapter.values())}, max: {max(per_chapter.values())}")


Loaded  'Pride and Prejudice' by Jane Austen
  Chunks  : 996
Chunks in chapter 1: 8
  Chunk chars — min: 19, median: 594, mean: 712, max: 3931
  Chunks/chapter — min: 6, median: 14, max: 38


## 4. Build the embedding index
All chapters are embedded once into a single FAISS flat-IP index.  
The **bounded vs. unbounded** distinction is enforced at *query time* via a  
pre-retrieval metadata filter — not by maintaining separate indices.  
This mirrors production vector stores (Pinecone, Weaviate, Bedrock Knowledge Bases).


In [11]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print(f"Loading embedding model: {EMBED_MODEL} …")
embedder   = SentenceTransformer(EMBED_MODEL)
texts      = [c.text for c in chunks]
print(f"Embedding {len(texts)} chapters …")
embeddings = embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True)
embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)   # cosine similarity via inner product

dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"FAISS index: {index.ntotal} vectors, {dim}d")


Loading embedding model: all-MiniLM-L6-v2 …


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding 996 chapters …


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

FAISS index: 996 vectors, 384d


## 5. Retrieval functions
**Bounded retriever** — two-stage:
1. Retrieve `top_k × 10` candidates (oversampling).
2. Drop any candidate with `chapter_index > reader_position` — the *hard anti-spoiler filter*.
3. Return top `top_k` survivors.

This is the architectural load-bearer: the model physically cannot see post-position text, regardless of what the prompt says.

We also define a **smart retrieval**, which calls the LLM to identify if the query is entity-focused (specifically about a character, location, etc). 
If the LLM identifies that, instead of doing semantic matching via embeddings, we simply filter for chunks which have the entity string in it.


In [47]:
def embed_query(query: str) -> np.ndarray:
    vec = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(vec)
    return vec

def extract_entity(query: str) -> str | None:
    prompt = (
        "If this question is asking about one or more specific named characters, "
        "places, or concepts from a book, return the MOST IMPORTANT name to search "
        "for (just one — pick the most specific or least common one, e.g. 'Wickham' "
        "over 'Darcy' if both appear). If the question is not about any specific "
        "named entity, return exactly 'NONE'."
    )
    raw = llm.complete(prompt, query, max_tokens=20).strip()
    return None if raw.upper() == "NONE" else raw

def retrieve_smart(query: str, reader_pos: int = READER_POSITION,
                   top_k: int = TOP_K) -> list[Chunk]:
    entity = extract_entity(query)
    if entity:
        lexical = retrieve_lexical(entity, reader_pos)
        if lexical:
            return lexical
    # Fall back to embedding retrieval
    return retrieve_bounded(query, reader_pos, top_k)

def retrieve_lexical(entity: str, reader_pos: int, max_results: int = 10) -> list[Chunk]:
    """Substring match on entity across in-bounds chunks, spread across chapters."""
    matches = [
        c for c in chunks
        if c.chapter_index <= reader_pos and entity.lower() in c.text.lower()
    ]
    # Spread: at most 2 per chapter, prefer earlier mentions (intro context)
    by_chapter = {}
    for c in matches:
        by_chapter.setdefault(c.chapter_index, []).append(c)
    spread = []
    for chap in sorted(by_chapter.keys()):
        spread.extend(by_chapter[chap][:2])
    return spread[:max_results]

def retrieve_unbounded(query: str, top_k: int = TOP_K) -> list[Chunk]:
    """No position filter — naive full-book baseline."""
    vec = embed_query(query)
    _, indices = index.search(vec, top_k)
    return [chunks[i] for i in indices[0] if i < len(chunks)]


def retrieve_bounded(query: str, reader_pos: int = READER_POSITION,
                     top_k: int = TOP_K) -> list[Chunk]:
    """Hard pre-retrieval filter: only chapters ≤ reader_pos are eligible."""
    vec        = embed_query(query)
    oversample = min(top_k * 10, len(chunks))
    _, indices = index.search(vec, oversample)
    # The hard filter — this line is the core anti-spoiler mechanism
    filtered   = [chunks[i] for i in indices[0]
                  if i < len(chunks) and chunks[i].chapter_index <= reader_pos]
    return filtered[:top_k]


def format_context(retrieved: list[Chunk]) -> str:
    parts = []
    for c in retrieved:
        parts.append(
            f"[{c.chunk_id} | {c.chapter_label}, paragraph {c.paragraph_index} "
            f"(chapter position {c.chapter_index})]\n{c.text}"
        )
    return "\n\n---\n\n".join(parts)


# ── Sanity check ───────────────────────────────────────────────────────────
print("=== Sanity check: query about Mr. Darcy's first proposal ===")
print(f"(Occurs in Chapter 34; reader has read through Chapter {READER_POSITION})\n")

q = "Does Mr. Darcy propose to Elizabeth?"
bounded_res   = retrieve_bounded(q)
unbounded_res = retrieve_unbounded(q)
smart_res = retrieve_smart(q)

print(f"Bounded    retrieved chapters: {[c.chapter_index for c in bounded_res]}")
print(f"Unbounded  retrieved chapters: {[c.chapter_index for c in unbounded_res]}")

q2 = "Who is Mr. Wickham?"
smart_res2 = retrieve_smart(q2)
print(f"\nSmart on entity query: {[c.chapter_index for c in smart_res2]}")
# Should include chapter 15 with high probability

q3 = "Why does Darcy dislike Mr. Wickham?"

max_b = max((c.chapter_index for c in bounded_res), default=0)
print(f"\n{'✅ PASS' if max_b <= READER_POSITION else '❌ FAIL'} — bounded retriever respects reader position")


=== Sanity check: query about Mr. Darcy's first proposal ===
(Occurs in Chapter 34; reader has read through Chapter 15)

Bounded    retrieved chapters: [6, 6, 10, 9, 6, 6, 3, 3, 11, 6]
Unbounded  retrieved chapters: [59, 32, 59, 52, 59, 6, 33, 51, 53, 57]

Smart on entity query: [15, 15]

✅ PASS — bounded retriever respects reader position


## 6. QA function
The system prompt uses **positive, context-scoped framing** (Zhou et al. 2023):  
the model is told it *is* a bounded reading assistant, not told what to *avoid*.  
This sidesteps the ironic-rebound failure mode of prohibition-style instructions.


In [48]:
SYSTEM_QA = f"""You are a reading companion for "{BOOK_TITLE}" by {BOOK_AUTHOR}.
You are reading alongside the reader and only know what they have read so far.
The passages below are the portion of the book the reader has reached. Each passage is labeled with a chunk_id (e.g. ch06_p03).

Your job is to help them understand what they have read — discussing characters, events, themes, references, and meaning — without spoiling anything from later in the book.

KNOWLEDGE BOUNDARY
- For anything specific to this book's plot, characters, or events: use ONLY the provided passages. Do not draw on outside knowledge of the book, even if you recognize it.
- For general world knowledge — historical context, cultural references, vocabulary, literary conventions, what a "ball" or an "entail" is — you may answer from your own knowledge. The reader is not asking you to forget how the world works.

RESPONSE SPECTRUM
Choose the response that best fits what the passages actually support:

1. ANSWER FULLY when the passages contain a clear answer. Cite the chunk_ids you used.

2. ANSWER PARTIALLY when the passages support some of the answer but not all. State what you can say from the passages (with chunk_ids), then name what's missing — e.g. "From what I've read with you, I know X and Y, but I don't have anything yet about Z." Partial answers are good. Do not refuse just because the answer is incomplete.

3. DEFER only when answering even partially would either (a) require information that isn't in the passages at all, or (b) require you to hint at events the reader hasn't reached. In that case, say something like: "No information available up to this point.".

Lean toward answering. Defer only when partial information would itself be a spoiler or when the passages genuinely contain nothing relevant.

After answering the question, do not prompt further interaction. Be matter-of-fact when deferring.

Don't suggest the reader keep reading or speculate about what they'll find. State only what the passages do and don't show."""


def ask(question: str, retrieved: list[Chunk]) -> str:
    if not retrieved:
        return "No relevant passages found within the reader's current position."
    context  = format_context(retrieved)
    user_msg = (
        "PASSAGES FROM THE BOOK (reader's knowledge so far):\n"
        f"{context}\n\n"
        f"READER'S QUESTION: {question}"
    )
    return llm.complete(SYSTEM_QA, user_msg)


## 7. Generate eval question set (LLM-generated)
Questions are organised into three **spoiler-risk tiers**:

| Tier | Answer available | Expected behaviour |
|------|-----------------|-------------------|
| **safe** | Chapters 1–15 | Both pipelines should answer correctly |
| **boundary** | Chapters 13–17 | Edge cases; good for debugging |
| **spoiler** | Chapters 16–end | Bounded: should refuse. Unbounded: likely spoils |

The LLM generates questions; you review and edit them in **Section 7b** before running eval.


In [49]:
import json

SYSTEM_QGEN = (
    "You are creating evaluation questions for an anti-spoiler reading assistant test.\n"
    "Output ONLY valid JSON; no prose, no markdown fences, no explanation."
)

def generate_questions(tier: str, chapter_range: str, n: int) -> list[dict]:
    prompt = (
        f'The book is "{BOOK_TITLE}" by {BOOK_AUTHOR}.\n'
        f"Generate {n} questions a reader might ask about this book.\n\n"
        f"Tier: {tier}\n"
        f"The CORRECT ANSWER to each question first appears in {chapter_range}.\n\n"
        "Rules:\n"
        "- Each question must be answerable from the book text (not author biography)\n"
        "- Questions should feel natural, like something a reader would genuinely ask\n"
        f"- For 'spoiler' tier: answer must reveal something that happens AFTER chapter {READER_POSITION}\n"
        f"- For 'safe' tier: answer must be fully covered in chapters 1–{READER_POSITION}\n"
        "- For 'boundary' tier: answer is at the edge of the reader's current position\n\n"
        "Return a JSON array, each object with:\n"
        "  \"question\": string,\n"
        f"  \"tier\": \"{tier}\",\n"
        "  \"answer_appears_in\": \"Chapter N\" (your best estimate),\n"
        "  \"note\": one sentence explaining why this belongs in this tier\n\n"
        "Example (2 items):\n"
        '[{"question": "...", "tier": "' + tier + '", "answer_appears_in": "Chapter 3", "note": "..."}]'
    )
    raw = llm.complete(SYSTEM_QGEN, prompt)
    raw = raw.strip()
    if raw.startswith("```"):
        import re
        raw = re.sub(r"^```[\w]*\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"  ⚠️  JSON parse error (tier={tier}): {e}\n  Raw: {raw[:300]}")
        return []


print("Generating eval questions (3 tiers) …\n")
n           = N_QUESTIONS_PER_TIER
safe_qs     = generate_questions("safe",     f"chapters 1–{READER_POSITION}", n)
boundary_qs = generate_questions("boundary", f"chapters {READER_POSITION-2}–{READER_POSITION+2}", n)
spoiler_qs  = generate_questions("spoiler",  f"chapters {READER_POSITION+1}–end", n)
all_questions = safe_qs + boundary_qs + spoiler_qs

print(f"Generated  {len(safe_qs)} safe / {len(boundary_qs)} boundary / {len(spoiler_qs)} spoiler")
print(f"Total: {len(all_questions)} questions\n")
for q in all_questions:
    print(f"  [{q.get('tier','?').upper():8}] {q.get('question','')[:85]}")
    print(f"           → {q.get('answer_appears_in','?')}  |  {q.get('note','')[:75]}")
    print()


Generating eval questions (3 tiers) …

Generated  4 safe / 4 boundary / 4 spoiler
Total: 12 questions

  [SAFE    ] What is Elizabeth Bennet's first impression of Mr. Darcy when they meet at the ball?
           → Chapter 3  |  Elizabeth forms her initial negative opinion of Darcy at the Meryton ball e

  [SAFE    ] Why does Mr. Bingley come to reside in the neighborhood?
           → Chapter 1  |  The reason for Bingley's arrival at Netherfield is established in the openi

  [SAFE    ] What are Mrs. Bennet's main concerns about her daughters' futures?
           → Chapter 1  |  Mrs. Bennet's anxieties about marriage and entailment are clearly expressed

  [SAFE    ] How does Jane Bennet begin to connect with Mr. Bingley?
           → Chapter 9  |  The early stages of Jane and Bingley's acquaintance and mutual attraction d

  [BOUNDARY] What is Mr. Wickham's connection to Mr. Darcy?
           → Chapter 16  |  Wickham begins to reveal his history with Darcy in Chapter 16, providing cr


### 7b. Manual review pass
Inspect the generated questions. Edit, add, or remove entries directly in this cell,
then re-run it. The rest of the notebook consumes `questions`.

**What to check:**
- Does the tier assignment match the actual plot position?
- Are spoiler-tier questions genuinely revealing (not trivially deniable)?
- Are safe-tier questions fully answerable from chapters 1–15?

We add a definitional question as a test.


In [50]:
import copy

# Editable working copy — modify this list, then re-run downstream cells
questions = copy.deepcopy(all_questions)

# ── MANUAL EDITS GO HERE ──────────────────────────────────────────────────
#
# Change a question:
#   questions[0]["question"] = "Who is Elizabeth Bennet's father?"
#
# Add a question:
questions.append({
    "question": "Who is Mr. Wickham? Why is he important?",
    "tier": "definitional",
    "answer_appears_in": "",
    "note": ""
})

# Remove a question:
#   questions = [q for q in questions if "Darcy" not in q["question"]]
#
# ─────────────────────────────────────────────────────────────────────────

print(f"Final eval set: {len(questions)} questions")
for t in ("safe", "boundary", "spoiler", "definitional"):
    print(f"  {t:8}: {sum(1 for q in questions if q.get('tier')==t)}")


Final eval set: 13 questions
  safe    : 4
  boundary: 4
  spoiler : 4
  definitional: 1


## 8. LLM-as-judge: spoiler leakage detector
For each (question, answer) pair the judge classifies:

- `"safe"` — answer contains nothing beyond what a reader at chapter *N* would know
- `"leak"` — answer reveals plot details from chapters the reader hasn't reached
- `"refusal"` — assistant correctly declined to answer

The judge prompt is conservative: even indirect forward references  
(e.g. "Darcy's feelings change later") count as a leak.


In [51]:
SYSTEM_JUDGE = f"""You are a strict spoiler-safety judge for "{BOOK_TITLE}".
The reader has finished Chapter {READER_POSITION} of the book.

Classify the assistant's answer as exactly one of:

- "safe_full"    — gives a substantive answer using only information from chapters 1–{READER_POSITION}, or general world/cultural knowledge. No content from chapters {READER_POSITION + 1}+.

- "safe_partial" — gives a partial answer from in-bounds content AND explicitly flags what is missing or not yet covered (e.g. "I don't have anything yet about X"). This is correct behavior when the passages only partially answer the question.

- "safe_defer"   — declines to answer because the relevant content is genuinely not in chapters 1–{READER_POSITION}, AND a partial answer would itself spoil. Phrased as "we haven't gotten to that yet" or similar. This is correct behavior for true spoiler questions.

- "over_refusal" — declines to answer even though the passages from chapters 1–{READER_POSITION} contain enough information to answer at least partially. This is a failure mode: the answer was available and the assistant bailed.

- "leak"         — reveals events, facts, or character developments from chapters {READER_POSITION + 1} or later, whether explicitly or by strong implication.

To distinguish "safe_defer" from "over_refusal", consider: could a reader who has read chapters 1–{READER_POSITION} answer this question, even partially, from what they read? If yes and the assistant declined, it is "over_refusal".

Respond with ONLY a JSON object:
  {{"verdict": "safe_full"|"safe_partial"|"safe_defer"|"over_refusal"|"leak", "reason": "one sentence"}}
No prose, no markdown fences."""

def judge_answer(question: str, answer: str) -> dict:
    user_msg = f"QUESTION: {question}\n\nASSISTANT ANSWER: {answer}"
    raw = llm.complete(SYSTEM_JUDGE, user_msg, max_tokens=200)
    raw = raw.strip()
    if raw.startswith("```"):
        import re
        raw = re.sub(r"^```[\w]*\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        for v in ("leak", "safe", "refusal"):
            if v in raw.lower():
                return {"verdict": v, "reason": raw[:120]}
        return {"verdict": "unknown", "reason": raw[:120]}


## 9. Run the evaluation
Each question generates 4 LLM calls: 2 × QA (bounded + unbounded) + 2 × judge.  
With 12 questions: ~48 calls. Expect 2–5 min depending on backend.


In [52]:
import time

results = []

for i, q in enumerate(questions, 1):
    question = q["question"]
    tier     = q.get("tier", "?")
    print(f"[{i:2}/{len(questions)}] [{tier.upper():8}] {question[:68]}…")

    # Bounded pipeline
    b_chunks = retrieve_smart(question)
    b_answer = ask(question, b_chunks)
    b_judge  = judge_answer(question, b_answer)

    # Unbounded pipeline
    u_chunks = retrieve_unbounded(question)
    u_answer = ask(question, u_chunks)
    u_judge  = judge_answer(question, u_answer)

    results.append({
        "question"          : question,
        "tier"              : tier,
        "answer_appears_in" : q.get("answer_appears_in", "?"),
        "bounded_chapters"  : [c.chapter_index for c in b_chunks],
        "bounded_answer"    : b_answer,
        "bounded_verdict"   : b_judge.get("verdict", "?"),
        "bounded_reason"    : b_judge.get("reason",  ""),
        "unbounded_chapters": [c.chapter_index for c in u_chunks],
        "unbounded_answer"  : u_answer,
        "unbounded_verdict" : u_judge.get("verdict", "?"),
        "unbounded_reason"  : u_judge.get("reason",  ""),
    })

    bv = b_judge.get("verdict", "?")
    uv = u_judge.get("verdict", "?")
    print(f"           bounded={'✅' if bv != 'leak' else '❌'} {bv:8}  "
          f"unbounded={'✅' if uv != 'leak' else '⚠️ LEAK'} {uv}")
    time.sleep(0.3)

print(f"\n✅ Eval complete — {len(results)} questions")


[ 1/13] [SAFE    ] What is Elizabeth Bennet's first impression of Mr. Darcy when they m…
           bounded=✅ over_refusal  unbounded=✅ safe_full
[ 2/13] [SAFE    ] Why does Mr. Bingley come to reside in the neighborhood?…
           bounded=✅ safe_full  unbounded=✅ safe_full
[ 3/13] [SAFE    ] What are Mrs. Bennet's main concerns about her daughters' futures?…
           bounded=✅ safe_full  unbounded=✅ over_refusal
[ 4/13] [SAFE    ] How does Jane Bennet begin to connect with Mr. Bingley?…
           bounded=✅ safe_full  unbounded=⚠️ LEAK leak
[ 5/13] [BOUNDARY] What is Mr. Wickham's connection to Mr. Darcy?…
           bounded=✅ safe_partial  unbounded=⚠️ LEAK leak
[ 6/13] [BOUNDARY] Does Mr. Collins actually understand how ridiculous he seems?…
           bounded=✅ safe_full  unbounded=⚠️ LEAK leak
[ 7/13] [BOUNDARY] What does Elizabeth learn about Darcy's role in Bingley's life?…
           bounded=✅ over_refusal  unbounded=⚠️ LEAK leak
[ 8/13] [BOUNDARY] How does Jane respond to 

## 10. Results — side-by-side comparison table

In [53]:
import pandas as pd

ICONS = {"safe": "✅ safe", "leak": "❌ LEAK", "refusal": "🚫 refused", "unknown": "❓"}

df = pd.DataFrame([{
    "Tier"             : r["tier"],
    "Question"         : (r["question"][:72] + "…") if len(r["question"]) > 72 else r["question"],
    "Answer in"        : r["answer_appears_in"],
    "Bounded"          : ICONS.get(r["bounded_verdict"],   r["bounded_verdict"]),
    "Unbounded"        : ICONS.get(r["unbounded_verdict"], r["unbounded_verdict"]),
    "Bounded reason"   : r["bounded_reason"][:75],
    "Unbounded reason" : r["unbounded_reason"][:75],
} for r in results])

tier_order = {"spoiler": 0, "boundary": 1, "safe": 2}
df = (df.assign(_s=df["Tier"].map(tier_order).fillna(3))
        .sort_values("_s").drop(columns="_s").reset_index(drop=True))

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 60)
df


,Tier,Question,Answer in,Bounded,Unbounded,Bounded reason,Unbounded reason
0,spoiler,Does Jane Bennet end up with Mr. Bingley?,Chapter 55,over_refusal,❌ LEAK,"By chapter 15, readers know enough about Bingley's attentions to Jane, his","The assistant cites events from chapters 54 and 61, which are well beyond c"
1,spoiler,"Does Lydia Bennet's elopement with Mr. Wickham get resolved, and how doe…",Chapter 47,safe_defer,❌ LEAK,The assistant correctly declines to answer about Lydia's elopement and its,The assistant reveals that Lydia elopes with Wickham and that they marry (r
2,spoiler,What is revealed about Mr. Wickham's true character and past relationshi…,Chapter 34,safe_partial,❌ LEAK,The assistant correctly identifies that no explicit revelation occurs in ch,The answer explicitly reveals that Wickham eloped with Lydia Bennet (from c
3,spoiler,Does Elizabeth Bennet eventually accept Mr. Darcy's proposal of marriage…,Chapter 42,safe_defer,❌ LEAK,The assistant correctly declines to answer a question about future plot dev,"The assistant reveals events from Chapter 59 (well beyond Chapter 15), incl"
4,boundary,How does Jane respond to Bingley's sudden departure from Netherfield?,Chapter 15,safe_defer,❌ LEAK,"Bingley's departure from Netherfield occurs after Chapter 15, so the assist","The answer references events from chapters 21, 26, and 40, which are well b"
5,boundary,What does Elizabeth learn about Darcy's role in Bingley's life?,Chapter 17,over_refusal,❌ LEAK,"By Chapter 15, Elizabeth has learned from Wickham that Darcy was instrument","The answer cites chapters 16, 33, 54, and 12 as evidence, but the reader ha"
6,boundary,Does Mr. Collins actually understand how ridiculous he seems?,Chapter 15,safe_full,❌ LEAK,The answer uses only information from chapters 1–15 (explicit narrator comm,"The assistant cites ch20_p01 and ch57_p11, which are beyond chapter 15, rev"
7,boundary,What is Mr. Wickham's connection to Mr. Darcy?,Chapter 16,safe_partial,❌ LEAK,The assistant correctly identifies that chapters 1–15 establish tension bet,"The answer cites chapters 18, 21, 24, 25, and 52, which are all beyond chap"
8,safe,What is Elizabeth Bennet's first impression of Mr. Darcy when they meet …,Chapter 3,over_refusal,safe_full,Chapter 15 contains extensive information about Elizabeth's first impressio,The assistant accurately describes Elizabeth's first impression of Darcy fr
9,safe,Why does Mr. Bingley come to reside in the neighborhood?,Chapter 1,safe_full,safe_full,The assistant provides the complete explanation available in chapters 1–15:,The answer accurately explains Mr. Bingley's arrival at Netherfield using o


## 11. Aggregate metrics — spoiler-leakage rate

In [54]:
def rate(rs, key, verdict):
    return sum(1 for r in rs if r[key] == verdict) / len(rs) if rs else 0.0

rows = []
for tier in ("safe", "boundary", "spoiler", "ALL"):
    rs = results if tier == "ALL" else [r for r in results if r["tier"] == tier]
    rows.append({
        "Tier"               : tier,
        "N"                  : len(rs),
        "Bounded  leak"      : f"{rate(rs, 'bounded_verdict',   'leak'):.0%}",
        "Unbounded leak"     : f"{rate(rs, 'unbounded_verdict', 'leak'):.0%}",
        "Bounded  refusal"   : f"{rate(rs, 'bounded_verdict',   'refusal'):.0%}",
        "Unbounded refusal"  : f"{rate(rs, 'unbounded_verdict', 'refusal'):.0%}",
    })

metrics_df = pd.DataFrame(rows)
print(metrics_df.to_string(index=False))

print()
print("Interpretation:")
print("  Bounded   leak (spoiler tier) → target ≈ 0%.   Non-zero = architecture failing.")
print("  Unbounded leak (spoiler tier) → typically high. Shows the problem being solved.")
print("  Bounded refusal (safe tier)   → target ≈ 0%.   High = over-refusal (false positive).")
print("  Bounded refusal (spoiler tier)→ target ≈ 100%. Desired behaviour.")


    Tier  N Bounded  leak Unbounded leak Bounded  refusal Unbounded refusal
    safe  4            0%            25%               0%                0%
boundary  4            0%           100%               0%                0%
 spoiler  4            0%           100%               0%                0%
     ALL 13            0%            77%               0%                0%

Interpretation:
  Bounded   leak (spoiler tier) → target ≈ 0%.   Non-zero = architecture failing.
  Unbounded leak (spoiler tier) → typically high. Shows the problem being solved.
  Bounded refusal (safe tier)   → target ≈ 0%.   High = over-refusal (false positive).
  Bounded refusal (spoiler tier)→ target ≈ 100%. Desired behaviour.


## 12. Qualitative drill-down
Inspect full answers for any question. Useful for qualitative analysis and debugging.


In [55]:
def show_result(substring: str):
    """Print full bounded + unbounded answers for questions matching the substring."""
    matches = [r for r in results if substring.lower() in r["question"].lower()]
    if not matches:
        print(f"No results matching {substring!r}")
        return
    for r in matches:
        print("=" * 70)
        print(f"QUESTION : {r['question']}")
        print(f"TIER     : {r['tier']}  |  answer first in: {r['answer_appears_in']}")
        print()
        print(f"── BOUNDED  (retrieved chapters {r['bounded_chapters']}) ──")
        print(r["bounded_answer"])
        print(f"→ {r['bounded_verdict'].upper()}  |  {r['bounded_reason']}")
        print()
        print(f"── UNBOUNDED (retrieved chapters {r['unbounded_chapters']}) ──")
        print(r["unbounded_answer"])
        print(f"→ {r['unbounded_verdict'].upper()}  |  {r['unbounded_reason']}")
        print()


# Example: inspect any question about Darcy
show_result("Darcy")


QUESTION : What is Elizabeth Bennet's first impression of Mr. Darcy when they meet at the ball?
TIER     : safe  |  answer first in: Chapter 3

── BOUNDED  (retrieved chapters [3, 6, 8, 8]) ──
No information available up to this point.

The passages show us *Darcy's* perspective on Elizabeth—he was standing near her during the ball and later admits to Miss Bingley that Elizabeth Bennet has inspired his reflections. But they don't contain Elizabeth's own thoughts or first impression of him at that initial meeting.
→ OVER_REFUSAL  |  Chapter 15 contains extensive information about Elizabeth's first impression of Darcy at the ball, including her observations of his behavior, appearance, and her negative judgments about his pride and coldness.

── UNBOUNDED (retrieved chapters [59, 53, 11, 9, 3, 40, 10, 3, 31, 8]) ──
From the passages, Elizabeth's first impression of Mr. Darcy is quite negative.

At the ball in Chapter 3, Elizabeth overhears Bingley trying to persuade Darcy to dance, with 

## 13. Save results

In [56]:
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M")
pd.DataFrame(results).to_csv(f"antispoiler_results_{ts}.csv", index=False)
metrics_df.to_csv(f"antispoiler_metrics_{ts}.csv", index=False)
print(f"Saved → antispoiler_results_{ts}.csv")
print(f"Saved → antispoiler_metrics_{ts}.csv")


Saved → antispoiler_results_20260602_1455.csv
Saved → antispoiler_metrics_20260602_1455.csv
